In [7]:
from pathlib import Path
from datetime import datetime
import pandas as pd

ROOT = Path().resolve().parents[1]
DATA_PATH = ROOT / "data" / "interim" / "airports.parquet"
REPORT_PATH = ROOT / "reports" / "validation_airports.txt"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)

report_sections = []


def record(section: list, line: str = "") -> None:
    """Print a line and store it in the section list."""
    print(line)
    section.append(line)


print("Setup complete.")
print(f"Loaded: {DATA_PATH}")
print(f"Shape:  {df.shape}")

Setup complete.
Loaded: C:\Users\hp\Desktop\Term 7\Data Science\Project\flight-delay-predictor\data\interim\airports.parquet
Shape:  (171, 10)


In [8]:
section = []
record(section, "=" * 55)
record(section, "CHECK 1 — Shape")
record(section, "=" * 55)
record(section, f"  Rows:    {len(df):,}")
record(section, f"  Columns: {df.shape[1]}")

if len(df) == 0:
    record(section, "  FAIL — table is empty")
elif len(df) < 100:
    record(section, "  WARN — fewer than 100 airports, expected ~171")
else:
    record(section, "  PASS — row count looks correct")

report_sections.append(section)

CHECK 1 — Shape
  Rows:    171
  Columns: 10
  PASS — row count looks correct


In [9]:
EXPECTED_COLUMNS = [
    "airport_code",
    "airport_name",
    "city",
    "state",
    "latitude",
    "longitude",
    "elevation_ft",
    "airport_type",
    "num_runways",
    "timezone",
]

section = []
record(section, "=" * 55)
record(section, "CHECK 2 — Schema")
record(section, "=" * 55)

missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]

if missing_cols:
    record(section, f"  FAIL — missing columns: {missing_cols}")
else:
    record(section, f"  PASS — all {len(EXPECTED_COLUMNS)} expected columns present")

if extra_cols:
    record(section, f"  INFO — extra columns: {extra_cols}")

record(section, "")
record(section, "  Column dtypes:")
for col in df.columns:
    record(section, f"    {col:<22} {str(df[col].dtype)}")

report_sections.append(section)

CHECK 2 — Schema
  PASS — all 10 expected columns present

  Column dtypes:
    airport_code           str
    airport_name           str
    city                   str
    state                  str
    latitude               float64
    longitude              float64
    elevation_ft           float64
    airport_type           str
    num_runways            int64
    timezone               str


In [10]:
section = []
record(section, "=" * 55)
record(section, "CHECK 3 — Missing values")
record(section, "=" * 55)

total_missing = df.isnull().sum().sum()

if total_missing == 0:
    record(section, "  PASS — no missing values in any column")
else:
    record(section, f"  WARN — {total_missing} total missing values:")
    for col in df.columns:
        n = df[col].isnull().sum()
        if n > 0:
            pct = 100 * n / len(df)
            record(section, f"    {col:<22} {n:>4} missing  ({pct:.1f}%)")

report_sections.append(section)

CHECK 3 — Missing values
  PASS — no missing values in any column


In [11]:
section = []
record(section, "=" * 55)
record(section, "CHECK 4 — Duplicates")
record(section, "=" * 55)

exact_dups = df.duplicated().sum()
code_dups = df.duplicated(subset=["airport_code"]).sum()

if exact_dups == 0:
    record(section, "  PASS — no exact duplicate rows")
else:
    record(section, f"  FAIL — {exact_dups} exact duplicate rows")

if code_dups == 0:
    record(section, "  PASS — airport_code is unique (primary key valid)")
else:
    record(section, f"  FAIL — {code_dups} duplicate airport_code values:")
    dupes = df[df.duplicated(subset=["airport_code"], keep=False)]
    for line in (
        dupes[["airport_code", "airport_name"]].to_string(index=False).split("\n")
    ):
        record(section, f"    {line}")

report_sections.append(section)

CHECK 4 — Duplicates
  PASS — no exact duplicate rows
  PASS — airport_code is unique (primary key valid)


In [12]:
REQUIRED_ORIGIN_AIRPORTS = {
    "BOS",
    "CLT",
    "DEN",
    "DFW",
    "DTW",
    "EWR",
    "IAH",
    "JFK",
    "LAX",
    "MIA",
    "MSP",
    "ORD",
    "PHX",
    "SEA",
    "SFO",
}

section = []
record(section, "=" * 55)
record(section, "CHECK 5 — Airport code validity")
record(section, "=" * 55)

invalid_format = df[~df["airport_code"].astype(str).str.fullmatch(r"[A-Z]{3}")]
if len(invalid_format) == 0:
    record(section, "  PASS — all codes are valid 3-letter uppercase IATA format")
else:
    record(section, f"  FAIL — {len(invalid_format)} invalid airport codes:")
    for code in invalid_format["airport_code"].tolist():
        record(section, f"    - {code}")

present = set(df["airport_code"])
missing = REQUIRED_ORIGIN_AIRPORTS - present
if not missing:
    record(section, "  PASS — all 15 origin airports present in table")
else:
    record(section, f"  FAIL — {len(missing)} origin airports missing:")
    for code in sorted(missing):
        record(section, f"    - {code}")

report_sections.append(section)

CHECK 5 — Airport code validity
  PASS — all codes are valid 3-letter uppercase IATA format
  PASS — all 15 origin airports present in table


In [13]:
LAT_MIN, LAT_MAX = 17.0, 72.0
LON_MIN, LON_MAX = -180.0, -60.0

section = []
record(section, "=" * 55)
record(section, "CHECK 6 — Coordinate validity")
record(section, "=" * 55)

invalid_lat = df[(df["latitude"] < LAT_MIN) | (df["latitude"] > LAT_MAX)]
invalid_lon = df[(df["longitude"] < LON_MIN) | (df["longitude"] > LON_MAX)]

if len(invalid_lat) == 0:
    record(section, f"  PASS — all latitudes within US bounds ({LAT_MIN} to {LAT_MAX})")
else:
    record(section, f"  FAIL — {len(invalid_lat)} airports outside latitude bounds:")
    for _, row in invalid_lat.iterrows():
        record(section, f"    {row['airport_code']}  lat={row['latitude']}")

if len(invalid_lon) == 0:
    record(
        section, f"  PASS — all longitudes within US bounds ({LON_MIN} to {LON_MAX})"
    )
else:
    record(section, f"  FAIL — {len(invalid_lon)} airports outside longitude bounds:")
    for _, row in invalid_lon.iterrows():
        record(section, f"    {row['airport_code']}  lon={row['longitude']}")

record(section, "")
record(
    section,
    f"  Latitude  range: {df['latitude'].min():.4f}  to  {df['latitude'].max():.4f}",
)
record(
    section,
    f"  Longitude range: {df['longitude'].min():.4f}  to  {df['longitude'].max():.4f}",
)

report_sections.append(section)

CHECK 6 — Coordinate validity
  PASS — all latitudes within US bounds (17.0 to 72.0)
  PASS — all longitudes within US bounds (-180.0 to -60.0)

  Latitude  range: 17.7014  to  64.8151
  Longitude range: -159.3371  to  -64.8026


In [14]:
section = []
record(section, "=" * 55)
record(section, "CHECK 7 — Elevation validity")
record(section, "=" * 55)

null_elev = df["elevation_ft"].isna().sum()
negative_elev = df[df["elevation_ft"] < 0]

if null_elev == 0:
    record(section, "  PASS — no null elevations")
else:
    record(section, f"  WARN — {null_elev} null elevations")

if len(negative_elev) == 0:
    record(section, "  PASS — no negative elevations")
else:
    record(section, f"  INFO — {len(negative_elev)} airports below sea level (valid):")
    for _, row in negative_elev.iterrows():
        record(section, f"    {row['airport_code']}  {row['elevation_ft']} ft")

record(section, "")
record(section, f"  Min elevation:  {df['elevation_ft'].min():.0f} ft")
record(section, f"  Max elevation:  {df['elevation_ft'].max():.0f} ft")
record(section, f"  Mean elevation: {df['elevation_ft'].mean():.0f} ft")

report_sections.append(section)

CHECK 7 — Elevation validity
  PASS — no null elevations
  PASS — no negative elevations

  Min elevation:  3 ft
  Max elevation:  7680 ft
  Mean elevation: 1066 ft


In [15]:
VALID_TYPES = {
    "large_airport",
    "medium_airport",
    "small_airport",
    "heliport",
    "seaplane_base",
    "closed",
}

section = []
record(section, "=" * 55)
record(section, "CHECK 8 — Airport type distribution")
record(section, "=" * 55)

type_counts = df["airport_type"].value_counts()
invalid_types = df[~df["airport_type"].isin(VALID_TYPES)]

record(section, "  Distribution:")
for atype, count in type_counts.items():
    pct = 100 * count / len(df)
    record(section, f"    {atype:<22} {count:>4}  ({pct:.1f}%)")

if len(invalid_types) == 0:
    record(section, "  PASS — all airport_type values are valid")
else:
    record(section, f"  WARN — {len(invalid_types)} unexpected type values:")
    for _, row in invalid_types.iterrows():
        record(section, f"    {row['airport_code']}  {row['airport_type']}")

report_sections.append(section)

CHECK 8 — Airport type distribution
  Distribution:
    large_airport            93  (54.4%)
    medium_airport           77  (45.0%)
    small_airport             1  (0.6%)
  PASS — all airport_type values are valid


In [16]:
section = []
record(section, "=" * 55)
record(section, "CHECK 9 — Runway counts")
record(section, "=" * 55)

zero_runways = df[df["num_runways"] == 0]
negative_runways = df[df["num_runways"] < 0]

if len(negative_runways) == 0:
    record(section, "  PASS — no negative runway counts")
else:
    record(section, f"  FAIL — {len(negative_runways)} negative runway counts")

if len(zero_runways) == 0:
    record(section, "  PASS — all airports have at least 1 runway")
else:
    record(section, f"  WARN — {len(zero_runways)} airports with 0 runways:")
    for _, row in zero_runways.iterrows():
        record(section, f"    {row['airport_code']}  {row['airport_name']}")

record(section, "")
record(section, f"  Min runways:  {df['num_runways'].min()}")
record(section, f"  Max runways:  {df['num_runways'].max()}")
record(section, f"  Mean runways: {df['num_runways'].mean():.1f}")

report_sections.append(section)

CHECK 9 — Runway counts
  PASS — no negative runway counts
  PASS — all airports have at least 1 runway

  Min runways:  1
  Max runways:  11
  Mean runways: 2.8


In [17]:
section = []
record(section, "=" * 55)
record(section, "CHECK 10 — Timezones")
record(section, "=" * 55)

null_tz = df["timezone"].isna().sum()
unknown_tz = df[df["timezone"] == "Unknown"]

if null_tz == 0 and len(unknown_tz) == 0:
    record(section, "  PASS — all airports have a valid IANA timezone")
else:
    if null_tz > 0:
        record(section, f"  FAIL — {null_tz} null timezones")
    if len(unknown_tz) > 0:
        record(section, f"  WARN — {len(unknown_tz)} airports with Unknown timezone:")
        for _, row in unknown_tz.iterrows():
            record(section, f"    {row['airport_code']}")

record(section, "")
record(section, "  Timezone distribution:")
for tz, count in df["timezone"].value_counts().items():
    record(section, f"    {tz:<35} {count:>4}")

report_sections.append(section)

CHECK 10 — Timezones
  PASS — all airports have a valid IANA timezone

  Timezone distribution:
    America/New_York                      59
    America/Chicago                       44
    America/Los_Angeles                   26
    America/Denver                        18
    Pacific/Honolulu                       4
    America/Detroit                        3
    America/Sitka                          3
    America/Puerto_Rico                    3
    America/Indiana/Indianapolis           2
    America/Phoenix                        2
    America/Anchorage                      2
    America/St_Thomas                      2
    America/Boise                          1
    America/Kentucky/Louisville            1
    America/Juneau                         1


In [18]:
ALL_FLIGHT_AIRPORTS = {
    "ABQ",
    "ACK",
    "AGS",
    "ALB",
    "AMA",
    "ANC",
    "ATL",
    "ATW",
    "AUS",
    "AVL",
    "AVP",
    "BDL",
    "BFL",
    "BGR",
    "BHM",
    "BIL",
    "BIS",
    "BNA",
    "BOI",
    "BOS",
    "BQN",
    "BTR",
    "BTV",
    "BUF",
    "BUR",
    "BWI",
    "BZN",
    "CAE",
    "CHA",
    "CHS",
    "CID",
    "CLE",
    "CLT",
    "CMH",
    "COS",
    "CVG",
    "DAB",
    "DAL",
    "DAY",
    "DCA",
    "DEN",
    "DFW",
    "DLH",
    "DRO",
    "DSM",
    "DTW",
    "ECP",
    "EGE",
    "ELP",
    "EUG",
    "EWR",
    "EYW",
    "FAI",
    "FAR",
    "FAT",
    "FCA",
    "FLL",
    "FSD",
    "GEG",
    "GFK",
    "GJT",
    "GRB",
    "GRR",
    "GSO",
    "GSP",
    "GTF",
    "GUC",
    "HDN",
    "HNL",
    "HOU",
    "HRL",
    "HSV",
    "HYA",
    "IAD",
    "IAH",
    "ICT",
    "ILM",
    "IND",
    "ISP",
    "JAC",
    "JAX",
    "JFK",
    "JNU",
    "KOA",
    "KTN",
    "LAS",
    "LAX",
    "LBB",
    "LEX",
    "LGA",
    "LIH",
    "LIT",
    "LNK",
    "MAF",
    "MCI",
    "MCO",
    "MDT",
    "MDW",
    "MEM",
    "MFE",
    "MFR",
    "MHT",
    "MIA",
    "MKE",
    "MRY",
    "MSN",
    "MSO",
    "MSP",
    "MSY",
    "MTJ",
    "MVY",
    "MYR",
    "OAK",
    "OGG",
    "OKC",
    "OMA",
    "ONT",
    "ORD",
    "ORF",
    "ORH",
    "PAE",
    "PBI",
    "PDX",
    "PHL",
    "PHX",
    "PIT",
    "PNS",
    "PQI",
    "PSC",
    "PSE",
    "PSP",
    "PVD",
    "PWM",
    "RAP",
    "RDM",
    "RDU",
    "RIC",
    "RNO",
    "ROC",
    "RST",
    "RSW",
    "SAN",
    "SAT",
    "SAV",
    "SBA",
    "SBN",
    "SBP",
    "SDF",
    "SEA",
    "SFO",
    "SIT",
    "SJC",
    "SJU",
    "SLC",
    "SMF",
    "SNA",
    "SRQ",
    "STL",
    "STS",
    "STT",
    "STX",
    "SYR",
    "TLH",
    "TPA",
    "TUL",
    "TUS",
    "TVC",
    "TYS",
    "VPS",
    "WRG",
    "XNA",
}

section = []
record(section, "=" * 55)
record(section, "CHECK 11 — Join coverage")
record(section, "=" * 55)

present_in_metadata = set(df["airport_code"])
not_covered = ALL_FLIGHT_AIRPORTS - present_in_metadata

record(section, f"  Flight dataset airports:   {len(ALL_FLIGHT_AIRPORTS)}")
record(
    section,
    f"  Covered by metadata:       {len(ALL_FLIGHT_AIRPORTS) - len(not_covered)}",
)
record(section, f"  Not covered:               {len(not_covered)}")

if not not_covered:
    record(section, "  PASS — metadata covers 100% of flight dataset airports")
else:
    record(
        section, f"  WARN — {len(not_covered)} flight airports have no metadata row:"
    )
    record(section, "  These will produce NULLs when joined to flight records.")
    for code in sorted(not_covered):
        record(section, f"    - {code}")

report_sections.append(section)

CHECK 11 — Join coverage
  Flight dataset airports:   171
  Covered by metadata:       171
  Not covered:               0
  PASS — metadata covers 100% of flight dataset airports


In [20]:
section = []
record(section, "=" * 55)
record(section, "CHECK 12 — Descriptive summary")
record(section, "=" * 55)

record(section, "  Airport type breakdown:")
for atype, count in df["airport_type"].value_counts().items():
    pct = 100 * count / len(df)
    record(section, f"    {atype:<22} {count:>4}  ({pct:.1f}%)")

record(section, "")
record(section, "  State coverage:")
record(section, f"    Unique states represented: {df['state'].nunique()}")
record(section, "    Top 5 states by airport count:")
for state, count in df["state"].value_counts().head(5).items():
    record(section, f"      {state}  {count}")

record(section, "")
record(section, "  Runway statistics:")
record(section, f"    Airports with 1 runway:    {(df['num_runways'] == 1).sum()}")
record(section, f"    Airports with 2 runways:   {(df['num_runways'] == 2).sum()}")
record(section, f"    Airports with 3+ runways:  {(df['num_runways'] >= 3).sum()}")
record(
    section,
    f"    Most runways: {df.loc[df['num_runways'].idxmax(), 'airport_code']} "
    f"({df['num_runways'].max()} runways)",
)

record(section, "")
record(section, "  Elevation statistics:")
record(
    section,
    f"    Highest airport: {df.loc[df['elevation_ft'].idxmax(), 'airport_code']} "
    f"({df['elevation_ft'].max():.0f} ft)",
)
record(
    section,
    f"    Lowest airport:  {df.loc[df['elevation_ft'].idxmin(), 'airport_code']} "
    f"({df['elevation_ft'].min():.0f} ft)",
)

record(section, "")
record(section, "  Geographic coverage:")
record(section, f"    Timezone zones covered: {df['timezone'].nunique()}")
for tz, count in df["timezone"].value_counts().items():
    record(section, f"      {tz:<35} {count} airports")

report_sections.append(section)

CHECK 12 — Descriptive summary
  Airport type breakdown:
    large_airport            93  (54.4%)
    medium_airport           77  (45.0%)
    small_airport             1  (0.6%)

  State coverage:
    Unique states represented: 50
    Top 5 states by airport count:
      CA  16
      FL  14
      TX  12
      CO  8
      NY  7

  Runway statistics:
    Airports with 1 runway:    19
    Airports with 2 runways:   62
    Airports with 3+ runways:  90
    Most runways: ORD (11 runways)

  Elevation statistics:
    Highest airport: GUC (7680 ft)
    Lowest airport:  EYW (3 ft)

  Geographic coverage:
    Timezone zones covered: 15
      America/New_York                    59 airports
      America/Chicago                     44 airports
      America/Los_Angeles                 26 airports
      America/Denver                      18 airports
      Pacific/Honolulu                    4 airports
      America/Detroit                     3 airports
      America/Sitka                       

In [21]:
header = [
    "=" * 55,
    "AIRPORTS METADATA VALIDATION REPORT",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"File:      {DATA_PATH}",
    f"Rows:      {len(df)}",
    f"Columns:   {df.shape[1]}",
    "=" * 55,
    "",
]

all_lines = header.copy()
for section in report_sections:
    all_lines.extend(section)
    all_lines.append("")

REPORT_PATH.write_text("\n".join(all_lines), encoding="utf-8")

print("=" * 55)
print("Report written successfully.")
print(f"Location: {REPORT_PATH}")
print(f"Total checks: {len(report_sections)}")
print("=" * 55)

Report written successfully.
Location: C:\Users\hp\Desktop\Term 7\Data Science\Project\flight-delay-predictor\reports\validation_airports.txt
Total checks: 13
